In [1]:
import seaborn as sns

In [2]:
df=sns.load_dataset('iris')
df.head(3)

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa


In [3]:
from sklearn.compose import ColumnTransformer 
from sklearn.preprocessing import OrdinalEncoder
col=ColumnTransformer(transformers=[("transform", OrdinalEncoder() , ["species"])] , remainder='passthrough')
new_df=col.fit_transform(df)

In [4]:
import pandas as pd
new_df=pd.DataFrame(new_df , columns=["species","sepal_length","sepal_width","petal_length","petal_width"])
new_df.head()

# ## or 
# from sklearn.preprocessing import LabelEncoder 
# le=LabelEncoder()
# df["species"]=le.fit_transform(df["species"])

,species,sepal_length,sepal_width,petal_length,petal_width
0,0.0,5.1,3.5,1.4,0.2
1,0.0,4.9,3.0,1.4,0.2
2,0.0,4.7,3.2,1.3,0.2
3,0.0,4.6,3.1,1.5,0.2
4,0.0,5.0,3.6,1.4,0.2


In [5]:
# new_df["species"].unique()

In [6]:
new_df = new_df[new_df['species'] != 0][["sepal_length","sepal_width",'species']]
new_df.head()

,sepal_length,sepal_width,species
50,7.0,3.2,1.0
51,6.4,3.2,1.0
52,6.9,3.1,1.0
53,5.5,2.3,1.0
54,6.5,2.8,1.0


In [7]:
X = new_df.iloc[:,0:2]
y = new_df.iloc[:,-1]

In [8]:
y

50     1.0
51     1.0
52     1.0
53     1.0
54     1.0
      ... 
145    2.0
146    2.0
147    2.0
148    2.0
149    2.0
Name: species, Length: 100, dtype: float64

In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
import numpy as np

### Cross_val_Score

In [10]:
estimators=[("lr" ,LogisticRegression()) , ("knn" , KNeighborsClassifier()) , ("rf" , RandomForestClassifier())]
cross_val_score(estimator=estimators[1][1], X=X,y=y , cv=10 ,scoring="accuracy")
# for all models and average cross val score
for name , model in estimators:
    score=cross_val_score(estimator=model, X=X,y=y , cv=10 , scoring="accuracy")
    print(name  , np.mean(score))

lr 0.75
knn 0.62
rf 0.58


### Hard Voting

In [11]:
from sklearn.ensemble import VotingClassifier 
vc= VotingClassifier(estimators=estimators , voting="hard")
score = cross_val_score(vc , X ,y , cv=10 , scoring="accuracy")
print(np.mean(score))

0.67


### Soft Voting

In [12]:
vc1= VotingClassifier(estimators=estimators , voting="soft")
score = cross_val_score(vc , X ,y , cv=10 , scoring="accuracy")
print(np.mean(score))

0.6799999999999999


### Weighted Voting

In [13]:
for i in range(1,3):
    for j in range(1,3):
        for k in range(1,3):
            vc = VotingClassifier(estimators=estimators,voting='soft',weights=[i,j,k])
            x = cross_val_score(vc,X,y,cv=10,scoring='accuracy')
            print("for i={},j={},k={}".format(i,j,k),np.round(np.mean(x),2))

for i=1,j=1,k=1 0.66
for i=1,j=1,k=2 0.64
for i=1,j=2,k=1 0.65
for i=1,j=2,k=2 0.64
for i=2,j=1,k=1 0.68
for i=2,j=1,k=2 0.66
for i=2,j=2,k=1 0.67
for i=2,j=2,k=2 0.65


### Classification of same Algorithm

In [14]:
import numpy as np
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import cross_val_score
from sklearn.datasets import make_classification
X, y = make_classification(n_samples=1000, n_features=20, n_informative=15, n_redundant=5, random_state=2)
estimators = [
    ('svm1', CalibratedClassifierCV(SVC(kernel='poly', degree=1), ensemble=False)),
    ('svm2', CalibratedClassifierCV(SVC(kernel='poly', degree=2), ensemble=False)),
    ('svm3', CalibratedClassifierCV(SVC(kernel='poly', degree=3), ensemble=False)),
    ('svm4', CalibratedClassifierCV(SVC(kernel='poly', degree=4), ensemble=False)),
    ('svm5', CalibratedClassifierCV(SVC(kernel='poly', degree=5), ensemble=False))
]
for name, model in estimators:
    scores = cross_val_score(model, X, y, cv=10, scoring='accuracy')
    print(name, np.round(np.mean(scores), 2))  

svm1 0.85
svm2 0.86
svm3 0.89
svm4 0.85
svm5 0.87


In [15]:
vc1 = VotingClassifier(estimators=estimators,voting='soft')
x = cross_val_score(vc1,X,y,cv=10,scoring='accuracy')
print(np.round(np.mean(x),2))

0.93


### Voting Regressor

In [16]:
import pandas as pd
import numpy as np
data_url = "http://lib.stat.cmu.edu/datasets/boston"
raw_df = pd.read_csv(data_url, sep="\s+", skiprows=22, header=None)

X = np.hstack([raw_df.values[::2, :], raw_df.values[1::2, :2]])
y = raw_df.values[1::2, 2]

<>:4: SyntaxWarning: invalid escape sequence '\s'
<>:4: SyntaxWarning: invalid escape sequence '\s'
C:\Users\LENOVO\AppData\Local\Temp\ipykernel_16444\3441816252.py:4: SyntaxWarning: invalid escape sequence '\s'
  raw_df = pd.read_csv(data_url, sep="\s+", skiprows=22, header=None)


In [17]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.model_selection import cross_val_score

In [18]:
lr = LinearRegression()
dt = DecisionTreeRegressor()
svr = SVR() 
estimators = [('lr',lr),('dt',dt),('svr',svr)]

In [19]:
for estimator in estimators:
  scores = cross_val_score(estimator[1],X,y,scoring='r2',cv=10)
  print(estimator[0],np.round(np.mean(scores),2)) 

lr 0.2
dt -0.11
svr -0.41


In [20]:
from sklearn.ensemble import VotingRegressor 
vr = VotingRegressor(estimators)
scores = cross_val_score(vr,X,y,scoring='r2',cv=10)
print("Voting Regressor",np.round(np.mean(scores),2))

Voting Regressor 0.4
